# Summary — comparison table

Best model per category, ranked by the cross-cohort **CAMP-only Balanced AUC**
(64v64, 100-rep bootstrap). Data is read from saved prediction CSVs.


## Setup

In [ ]:
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import roc_auc_score
from sklearn.utils import resample
from scipy.stats import ttest_ind

ROOT = Path.cwd()  # run from 09_ptrs-unified_model-evaluation/
PRED = ROOT / 'data' / 'predictions'
FIGDIR = ROOT / 'figures'
FIGDIR.mkdir(parents=True, exist_ok=True)
P_VALS = ['5e-05', '5e-04', '0_005', '0_05']
N_REPEATS = 100


def balanced_boot_auc(scores, y_true, n_repeats=N_REPEATS):
    """64v64-style balanced AUC: equal-sized case/control draws, 100 reps."""
    s = np.asarray(scores, dtype=float)
    y = np.asarray(y_true).astype(int)
    case = np.where(y == 1)[0]
    ctrl = np.where(y == 0)[0]
    n = min(len(case), len(ctrl))
    aucs = []
    for seed in range(n_repeats):
        cs = resample(case, n_samples=n, replace=False, random_state=seed)
        ct = (resample(ctrl, n_samples=n, replace=False, random_state=seed)
              if len(ctrl) > n else ctrl)
        sel = np.concatenate([cs, ct])
        aucs.append(roc_auc_score(y[sel], s[sel]))
    return float(np.mean(aucs)), float(np.std(aucs))


def load_camp_only(path):
    """Read a best_consistent__* prediction file in either saved format."""
    d = pd.read_csv(path)
    cols = set(d.columns)
    if {'y_pred', 'y_true'} <= cols:                       # per-cohort format
        return pd.DataFrame({'predicted_value': d['y_pred'].astype(float).values,
                             'asthma': d['y_true'].astype(int).values})
    if {'score', 'y_true', 'cohort'} <= cols:              # all-cohort format
        d = d[d['cohort'] == 'CAMP_only']
        return pd.DataFrame({'predicted_value': d['score'].astype(float).values,
                             'asthma': d['y_true'].astype(int).values})
    raise ValueError(f'unknown prediction-file format: {path}')


## 1. Comparison table

One row per category — the best option within it, ranked by CAMP-only Balanced AUC.
`P_VAL` is the TWAS P+T threshold (`1` = the standard PIP-filtered run, `-` = N/A).
`Delta_AUC` is the AUC gain over the **PRS-CSx (best)** baseline (set to 0).

In [ ]:
rows = []

# Rows 1-2: best PRS-CS / PRS-CSx (logistic-regression baseline) — altPRS config
prs = pd.read_csv(PRED / 'prscs_evaluation' / 'prs_predictions.csv')
for method in ['PRS-CS', 'PRS-CSx']:
    best = None
    for config in sorted(prs['config'].unique()):
        sub = prs[(prs['method'] == method) & (prs['config'] == config) &
                  (prs['eval_set'] == 'CAMP-only')]
        if sub.empty:
            continue
        auc, std = balanced_boot_auc(sub['score'], sub['y_true'])
        if best is None or auc > best['AUC']:
            best = {'AUC': auc, 'AUC_std': std, 'config': config}
    rows.append({'Category': f'{method} (altPRS)', 'Features': '-', 'P_VAL': '-',
                 'Classifier': f"Logistic Regression ({best['config']})",
                 'CAMP_Balanced_AUC': best['AUC'], 'AUC_std': best['AUC_std']})

# Rows 3-4: best single-feature MA-FOCUS PTRS (P_VAL=1)
for mv, mv_label, feat in [('tissue', 'tissue', 'Esophagus_Mucosa'),
                           ('ct', 'CT', 'cd4_naive')]:
    cf = pd.read_csv(PRED / f'meta_model_{mv}' / 'consistent_features.csv')
    model = cf.loc[cf['Feature'] == feat, 'Model'].iloc[0]
    pdir = PRED / f'meta_model_{mv}' / 'predictions'
    cands = (sorted(pdir.glob(f'best_consistent__{feat}__*__CAMP_only.csv'))
             or sorted(pdir.glob(f'best_consistent__{feat}__*.csv')))
    df = load_camp_only(cands[0])
    auc, std = balanced_boot_auc(df['predicted_value'], df['asthma'])
    rows.append({'Category': f'PTRS-{feat} ({mv_label}, MA-FOCUS, single feature)', 'Features': feat,
                 'P_VAL': '1', 'Classifier': model,
                 'CAMP_Balanced_AUC': auc, 'AUC_std': std})

# Rows 5-6: best single-feature TWAS P+T PTRS (LR-family only, across the 4 p-value runs)
LR_FAMILY_SAFE = {'Ridge_C_0_01', 'Ridge_C_0_1', 'Ridge_C_1_0',
                  'Lasso_C_0_1', 'Elastic_Net'}
SAFE_TO_PRETTY = {'Ridge_C_0_01': 'Ridge (C=0.01)', 'Ridge_C_0_1': 'Ridge (C=0.1)',
                  'Ridge_C_1_0':  'Ridge (C=1.0)',  'Lasso_C_0_1': 'Lasso (C=0.1)',
                  'Elastic_Net':  'Elastic Net'}
OTHER_COHORTS_SAFE = {'GACRS_test', 'CAMP_GTEx', 'CAMP_1KG', 'CAMP+1KG',
                      'GACRS_train_OOF'}
for mv, mv_label in [('tissue', 'tissue'), ('ct', 'CT')]:
    best = None
    for pv in P_VALS:
        pdir = PRED / f'meta_model_{mv}__pval-{pv}' / 'predictions'
        if not pdir.exists(): continue
        seen = set()
        for f in sorted(pdir.glob('best_consistent__*.csv')):
            parts = f.stem.split('__')
            if len(parts) < 3: continue
            feat, model_safe = parts[1], parts[2]
            cohort = parts[3] if len(parts) >= 4 else None
            if cohort in OTHER_COHORTS_SAFE: continue
            if model_safe not in LR_FAMILY_SAFE: continue
            key = (feat, model_safe)
            if key in seen: continue
            seen.add(key)
            d = load_camp_only(f)
            if d.empty: continue
            a, s = balanced_boot_auc(d['predicted_value'], d['asthma'])
            if best is None or a > best['auc']:
                best = {'auc': a, 'std': s, 'feat': feat, 'model_safe': model_safe, 'pv': pv}
    if best is None: continue
    rows.append({
        'Category':  f'PTRS-{best["feat"]} ({mv_label}, TWAS P+T, single feature)',
        'Features':  best['feat'], 'P_VAL': best['pv'],
        'Classifier': SAFE_TO_PRETTY.get(best['model_safe'], best['model_safe']),
        'CAMP_Balanced_AUC': best['auc'], 'AUC_std': best['std'],
    })

# Unified PTRS tissue / CT (P_VAL=1, 7-model sweep) — meta-model exploration (unchanged)
for mv, label in [('tissue', 'Unified PTRS tissue (MA-FOCUS, meta-model)'),
                   ('ct', 'Unified PTRS CT (MA-FOCUS, meta-model)')]:
    r = pd.read_csv(PRED / f'meta_model_{mv}' / 'unified_balanced_results.csv')
    b = r.loc[r['AUC'].idxmax()]
    cf = pd.read_csv(PRED / f'meta_model_{mv}' / 'consistent_features.csv')
    rows.append({'Category': label, 'Features': ', '.join(cf['Feature'].tolist()),
                 'P_VAL': '1', 'Classifier': b['Method'],
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# ============================================================
# Rows 11-13: NEW — cross-modal integration (cd4_naive + Esophagus_Mucosa)
# per-feature OOF + altPRS PRS config (PRS-CS ϕ=auto + PRS-CSx ϕ=auto/META)
# ============================================================
integ = pd.read_csv(PRED / 'integrated_ptrs_prs_combined' / 'all_results.csv')
ic = integ[integ['Eval_Set'] == 'CAMP-only Balanced']

# Row 11: cross-modal PTRS only (per-feature OOF, no PRS) — best of the two anchors
feat_only = ic[ic['Approach'] == 'Per-feature only']
if not feat_only.empty:
    b = feat_only.loc[feat_only['AUC'].idxmax()]
    rows.append({'Category': 'Per-feature OOF PTRS (cross-modal, best alone)',
                 'Features': 'cd4_naive, Esophagus_Mucosa', 'P_VAL': '1',
                 'Classifier': b['Method'].replace(' alone (per-feature OOF)', '') + ' OOF',
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# Rows 12-13: best Direct integration per PRS variant
for prs_label in ['PRS-CS', 'PRS-CSx']:
    sub = ic[(ic['Approach'] == 'Direct') & (ic['Method'].str.contains(f'\\({prs_label}\\)', regex=True))]
    if sub.empty: continue
    b = sub.loc[sub['AUC'].idxmax()]
    # Strip the "Direct (PRS-CS) + " prefix from the method name for cleaner display
    classifier = b['Method'].replace(f'Direct ({prs_label}) + ', '')
    rows.append({'Category': f'Cross-modal + {prs_label} (Direct integration, altPRS)',
                 'Features': 'cd4_naive, Esophagus_Mucosa', 'P_VAL': '1',
                 'Classifier': classifier,
                 'CAMP_Balanced_AUC': b['AUC'], 'AUC_std': b['AUC_std']})

# ============================================================
# Assemble + save
# ============================================================
summary_table = pd.DataFrame(rows)
# Delta AUC vs PRS-CSx altPRS baseline (set to 0)
ref_auc = summary_table.loc[summary_table['Category'] == 'PRS-CSx (altPRS)',
                            'CAMP_Balanced_AUC'].iloc[0]
summary_table['Delta_AUC'] = summary_table['CAMP_Balanced_AUC'] - ref_auc
for c in ['CAMP_Balanced_AUC', 'AUC_std', 'Delta_AUC']:
    summary_table[c] = summary_table[c].round(4)
summary_table = summary_table[['Category', 'Features', 'P_VAL', 'Classifier',
                               'CAMP_Balanced_AUC', 'AUC_std', 'Delta_AUC']]
summary_table.to_csv(FIGDIR / 'summary_comparison_table.csv', index=False)
print(f"Saved -> {FIGDIR / 'summary_comparison_table.csv'}  "
      f"(Delta_AUC baseline: PRS-CSx altPRS = {ref_auc:.4f})\n")
print(summary_table.to_string(index=False))
